# Imperia Matching TFM — Demo interactiva

**Sistema híbrido de recomendación inmobiliaria explicable**

Este notebook ejecuta el TFM completo paso a paso:

| Modelo | Descripción |
|--------|-------------|
| **M0** | Baseline estructurado con pesos fijos |
| **M1** | Pesos optimizados sobre etiquetas |
| **M2** | Clientes con datos incompletos + extractor de notas |
| **M3** | M2 con pesos adaptados al contexto degradado |
| **M4** | Híbrido estructurado + embeddings semánticos (Transformer) |
| **M5** | RAG: retrieval M4 + razonamiento Claude (LLM) |

**Tiempo estimado de ejecución: ~3 minutos desde cero.**

## 1. Instalación y clonado del repositorio

In [ ]:
# Clonar el repositorio
!git clone https://github.com/Briian11/imperia-matching-tfm.git
%cd imperia-matching-tfm

In [ ]:
# Instalar dependencias
# sentence-transformers es mas ligero en Colab porque torch ya viene preinstalado
!pip install sentence-transformers anthropic --quiet
print('Dependencias instaladas.')

In [ ]:
import sys
sys.path.insert(0, 'src')
print('Entorno listo.')

## 2. Tests — verificar que todo funciona

In [ ]:
!python3 run_tests.py

## 3. Demo M0 — baseline estructurado

Dado un cliente, el sistema puntúa las 100 propiedades y devuelve el top-K con desglose explicable por criterio.

In [ ]:
!python3 run_tfm_demo.py --top-k 3 --metric-k 3 2>&1 | head -80

## 4. Comparativa M0 → M4 — métricas cuantitativas

Evaluación con Precision@K, Recall@K y NDCG@K sobre 35 clientes y 100 propiedades.

In [ ]:
import json

results = json.load(open('reports/results/comparison_all.json'))

models = list(results.keys())
metrics = ['ndcg@3', 'ndcg@5', 'ndcg@10', 'recall@5', 'recall@10']

col = max(len(m) for m in models) + 2
header = f"{'Metrica':<14}" + "".join(f"{m:>{col}}" for m in models)
print("=" * len(header))
print("COMPARATIVA DE MODELOS")
print("=" * len(header))
print(header)
print("-" * len(header))
for metric in metrics:
    row = f"{metric:<14}"
    for model in models:
        val = results[model].get(metric, 0)
        row += f"{val:>{col}.4f}"
    print(row)
print("-" * len(header))
print("\nClientes: 35 | Propiedades: 100")

## 5. Figuras comparativas

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, path, title in zip(
    axes,
    ['reports/figures/ndcg_comparison.png', 'reports/figures/recall_comparison.png'],
    ['NDCG@K por modelo', 'Recall@K por modelo']
):
    ax.imshow(mpimg.imread(path))
    ax.axis('off')
    ax.set_title(title, fontsize=13)
plt.tight_layout()
plt.show()

fig2, axes2 = plt.subplots(1, 2, figsize=(16, 5))
for ax, path, title in zip(
    axes2,
    ['reports/figures/ndcg_degraded.png', 'reports/figures/weights_radar.png'],
    ['NDCG@K clientes degradados', 'Distribucion de pesos M0/M1/M3']
):
    ax.imshow(mpimg.imread(path))
    ax.axis('off')
    ax.set_title(title, fontsize=13)
plt.tight_layout()
plt.show()

## 6. M4 — matching inverso semántico

Dado una propiedad, rankea todos los clientes por afinidad estructurada + semántica.

In [ ]:
# Cambiar p_003 por cualquier ID de propiedad del dataset
!PYTHONPATH=src python3 scripts/run_reverse_match.py \
    --property-id p_003 \
    --model m4 \
    --top-k 5

## 7. M4 con clientes degradados — rescate semántico

Compara M0 vs M4 cuando los clientes tienen campos perdidos.
La capa semántica recupera información desde las notas en lenguaje natural.

In [ ]:
!PYTHONPATH=src python3 scripts/compare_degraded_match.py \
    --property-id p_003 \
    --top-k 8

## 8. M5 RAG — razonamiento con LLM (Claude)

M4 recupera los top-5 candidatos por cliente. Claude razona en lenguaje natural
cuál es el mejor match y detecta preferencias implícitas que el scoring estructurado no capta.

In [ ]:
import json

m5 = json.load(open('reports/results/m5_rag_results.json'))

# Mostrar 4 ejemplos representativos
for item in m5[:4]:
    llm = item['llm_response']
    print(f"{'='*70}")
    print(f"Cliente: {item['client_name']} ({item['client_id']})")
    print(f"Top candidatos M4: {[r['property_id'] for r in item['retrieval_top_k']]}")
    print(f"\nMejor match segun Claude: {llm.get('mejor_match', '?')}")
    print(f"Razonamiento:\n  {llm.get('razonamiento', '-')}")
    print(f"\nPreferencia implicita detectada:\n  {llm.get('preferencia_implicita', '-')}")
    print(f"\nPunto debil:\n  {llm.get('punto_debil', '-')}")

In [ ]:
# OPCIONAL: ejecutar M5 en vivo con tu API key
# Solo necesario si quieres regenerar los resultados

ANTHROPIC_API_KEY = ""  # pega tu key aqui si quieres ejecutar en vivo

if ANTHROPIC_API_KEY:
    import os
    os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
    !PYTHONPATH=src python3 scripts/run_m5_rag.py --limit 3
else:
    print("Mostrando resultados pre-computados (sin API key).")
    print(f"Total clientes procesados: {len(m5)}")

---

## Resumen del sistema

```
ENTRADA
  Cliente (perfil estructurado + notas)
  Propiedad
        │
  ┌─────▼──────────────────────────┐
  │  Extractor de notas (M2/M3)    │  regex sobre texto libre
  └─────┬──────────────────────────┘
        │
  ┌─────▼──────────────────────────┐
  │  Scoring estructurado (M0/M1)  │  pesos × criterios = score 0-100
  └─────┬──────────────────────────┘
        │         ┌────────────────────────────┐
        │  M4 ──► │ + Similitud semántica       │  Transformer 384-dim
        │         └────────────────────────────┘
        │         ┌────────────────────────────┐
        │  M5 ──► │ + Razonamiento LLM (Claude) │  RAG
        │         └────────────────────────────┘
        │
  ┌─────▼──────────────────────────┐
  │  Ranking explicable            │  top-K con desglose por criterio
  └────────────────────────────────┘
```

**Métricas de evaluación**: Precision@K, Recall@K, NDCG@K  
**Dataset**: 35 clientes × 100 propiedades × 2358 etiquetas de relevancia  
**Dependencias externas obligatorias**: ninguna (M5 usa resultados pre-computados)